# Prepare graph memory on Kaggle
Sample one OpenEQA episode, review its frames, and extract captions and scene graphs for Neo4j.


In [ ]:
# Configuration — edit this cell before running; no paid requests are enabled by default.
import os
from pathlib import Path
SMOKE = os.getenv("RVM_NOTEBOOK_SMOKE") == "1"  # synthetic offline software checks
PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
REPOSITORY_URL = 'https://github.com/AlirezaAK2000/robot-visual-memory.git'
REPOSITORY_REF = 'main'  # a branch, tag, or commit SHA; resolved commit is recorded in provenance
UPDATE_REPOSITORY = False  # reruns reuse the checkout; True explicitly fetches the ref again
WORK = Path(os.getenv("RVM_WORK", str(PROJECT_ROOT/'rvm-single-episode') if Path('/kaggle').exists() else str(PROJECT_ROOT/'artifacts/kaggle-single-episode'))).resolve()
CONFIG_FILE = None  # defaults to PROJECT_ROOT/config.example.yaml; optional attached edited YAML
ATTACHED_RGB = None  # Path('/kaggle/input/my-rgb/open-eqa-hm3d-frames-v0.tgz') or extracted root
ALLOW_RGB_DOWNLOAD = False  # enable only inside Kaggle with enough disk
RGB_PROVIDER = 'huggingface'  # pinned episode TAR mirror; original 'dropbox' link has expired
SAMPLING_REVIEWED = False  # review the sampled and original preview videos before paid extraction
RUN_PILOT = False  # first three sampled frames (all if fewer); needs Kaggle Secrets/env key
PILOT_REVIEWED = False  # set True AFTER reviewing the pilot output and prompt
RUN_FULL_EXTRACTION = False  # explicitly enable after pilot review; capped persistent budget applies
print('Mode:', 'SYNTHETIC SMOKE' if SMOKE else 'real OpenEQA preparation')


## Setup
Check out the repository in `/kaggle/working` and install dependencies. If the kernel requests a restart, keep the same WORK directory.


In [ ]:
"""Standard-library-only Git bootstrap embedded into the standalone Kaggle notebook.

The checkout lives directly in the target directory. In-place init/fetch/checkout also works
when Kaggle has already written files there. Git's normal checkout protections preserve
untracked files; no force/reset/clean operations are used.
"""

import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path


def loaded_dependency_versions():
    """Inspect already imported dependencies without importing any scientific libraries."""
    names = {
        "numpy": "numpy",
        "pandas": "pandas",
        "PIL": "Pillow",
    }
    return {
        distribution: str(module.__version__)
        for name, distribution in names.items()
        if (module := sys.modules.get(name)) is not None and getattr(module, "__version__", None) is not None
    }


def require_current_dependency_versions(loaded):
    """Stop after pip replaces an imported package; native extensions require a kernel restart."""
    mismatches = []
    for distribution, loaded_version in loaded.items():
        try:
            installed_version = version(distribution)
        except PackageNotFoundError:
            installed_version = "missing"
        if installed_version != loaded_version:
            mismatches.append(f"{distribution}: kernel={loaded_version}, installed={installed_version}")
    if mismatches:
        raise RuntimeError(
            "Dependencies changed while the kernel still has older modules loaded:\n"
            + "\n".join(mismatches)
            + "\nRestart the kernel, keep the same WORK directory, set UPDATE_REPOSITORY=False, "
            "and rerun from configuration. Existing downloads and extraction caches remain on disk. "
            "Do not reload NumPy in place."
        )


def checkout_repository(repository_url, ref, target, *, update=False):
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=True)
    environment = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}

    def git(*args, required=True):
        result = subprocess.run(
            ["git", "-C", str(target), *args],
            env=environment,
            text=True,
            capture_output=True,
        )
        if required and result.returncode:
            raise RuntimeError(
                "Git bootstrap failed. Check Internet access/repository access and the message below. "
                "Existing files were not force-overwritten.\n" + result.stderr.strip()
            )
        return result

    if (target / ".git").exists():
        actual_root = Path(git("rev-parse", "--show-toplevel").stdout.strip()).resolve()
        if actual_root != target:
            raise RuntimeError("PROJECT_ROOT must be the root of its own checkout")
        origin = git("remote", "get-url", "origin", required=False)
        if origin.returncode == 0 and origin.stdout.strip().removesuffix(".git") != str(
            repository_url
        ).removesuffix(".git"):
            raise RuntimeError("PROJECT_ROOT already belongs to another repository; its origin was preserved")
        if origin.returncode:
            git("remote", "add", "origin", repository_url)
    else:
        git("init", "--quiet")
        git("remote", "add", "origin", repository_url)

    head = git("rev-parse", "--verify", "HEAD", required=False)
    previous_ref = git("config", "--local", "--get", "rvm.bootstrapRef", required=False).stdout.strip()
    dirty = bool(git("status", "--porcelain", "--untracked-files=no").stdout.strip())
    if previous_ref and previous_ref != ref and not update:
        raise RuntimeError(
            "REPOSITORY_REF changed; set UPDATE_REPOSITORY=True to check out the new ref explicitly"
        )
    reused = head.returncode == 0 and previous_ref == ref and not update
    if not reused:
        if dirty:
            raise RuntimeError(
                "Tracked files have local edits; commit/stash them before requesting a repository update"
            )
        git("fetch", "--depth", "1", "--no-tags", "origin", ref)
        # This refuses conflicting untracked files and preserves unrelated outputs/caches.
        git("checkout", "--detach", "FETCH_HEAD")
        git("config", "--local", "rvm.bootstrapRef", ref)

    for relative in ["pyproject.toml", "config.example.yaml", "src/rvm/__init__.py"]:
        if not (target / relative).is_file():
            raise RuntimeError(f"Repository is missing expected project file: {relative}")
    return {
        "repository_url": repository_url,
        "requested_ref": ref,
        "commit": git("rev-parse", "HEAD").stdout.strip(),
        "previous_commit": head.stdout.strip() if head.returncode == 0 else None,
        "reused_checkout": reused,
        "tracked_changes": bool(git("status", "--porcelain", "--untracked-files=no").stdout.strip()),
    }

import sys
if SMOKE:
    # Offline verification explicitly uses the local source tree and never clones/downloads.
    SOURCE = next((p for p in [Path.cwd(), Path.cwd().parent] if (p/'pyproject.toml').exists()), None)
    if SOURCE is None:
        raise RuntimeError('Synthetic smoke mode requires the local project root')
    source_provenance = {'synthetic_smoke': True, 'repository_url': REPOSITORY_URL, 'commit': None}
else:
    SOURCE = PROJECT_ROOT.resolve()
    source_provenance = checkout_repository(REPOSITORY_URL, REPOSITORY_REF, SOURCE, update=UPDATE_REPOSITORY)
    if (source_provenance['previous_commit'] is not None
        and source_provenance['previous_commit'] != source_provenance['commit']
        and any(name == 'rvm' or name.startswith('rvm.') for name in sys.modules)):
        raise RuntimeError('Repository updated. Restart the kernel, set UPDATE_REPOSITORY=False, and run in order to load the new code.')
SOURCE = SOURCE.resolve()
os.chdir(SOURCE)
WORK.mkdir(parents=True, exist_ok=True)
print('Project root:', SOURCE)
print('Repository commit:', source_provenance['commit'], 'reused:', source_provenance.get('reused_checkout', False))
if not SMOKE:
    dependencies_before_install = loaded_dependency_versions()
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', '-r', 'requirements-kaggle.txt'], cwd=SOURCE)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.'], cwd=SOURCE)
    require_current_dependency_versions(dependencies_before_install)
sys.path.insert(0, str(SOURCE/'src'))
from rvm.io import load_config, read_json, atomic_json, digest, export_zip
from rvm.schemas import Frame, Question, Record
config = load_config(CONFIG_FILE or SOURCE/'config.example.yaml')
config['dataset']['input_path'] = str(ATTACHED_RGB) if ATTACHED_RGB else config['dataset']['input_path']
config['dataset']['allow_download'] = ALLOW_RGB_DOWNLOAD
config['dataset']['rgb_provider'] = RGB_PROVIDER
BUNDLE = WORK/'compact-bundle'
atomic_json(BUNDLE/'provenance/source-code.json', source_provenance)
print('Source ready; bundle:', BUNDLE)


## Inventory and episode selection
Count collected RGB histories and plot their frame distribution before selecting one episode.


In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Image as DisplayImage, Video
from rvm.preparation import (metadata_first, select_environments, locate_rgb_source, acquire_rgb,
                             discover_rgb, inspect_images, prepare_gallery, contact_sheet)
from rvm.episode_selection import inventory_episodes, save_inventory, make_preview_video
if config['dataset'].get('environment_selection') not in {'smallest', 'largest'} or config['dataset']['environments'] != 1:
    raise ValueError('This notebook requires smallest/largest selection and environments=1')
if SMOKE:
    from rvm.fixture import create_episode_collection
    source, qa_rows = create_episode_collection(WORK/'synthetic-collection')
    archive_info = {'synthetic': True}
else:
    qa_rows, archive_info = metadata_first(config, WORK)
    display(pd.DataFrame(qa_rows).assign(dataset=lambda d: d.episode_history.str.split('/').str[0])
            .groupby(['dataset', 'category']).size().rename('questions').to_frame())
    print('Metadata histories:', len({r['episode_history'] for r in qa_rows}))
    source = locate_rgb_source(config, WORK, archive_info, qa_rows=qa_rows)
    if 'rgb_acquisition' in archive_info:
        print('RGB provider:', archive_info['rgb_acquisition']['provider'])
        print('Mirror commit:', archive_info['rgb_acquisition']['revision'])
        print('Original Dropbox archive byte identity is unverified; actual mirror counts follow.')
inventory, inventory_summary = inventory_episodes(source, qa_rows, config)
histogram = save_inventory(inventory, inventory_summary, BUNDLE/'review', synthetic=SMOKE, selection_mode=config['dataset']['environment_selection'])
display(pd.DataFrame(inventory).sort_values(['frame_count','environment_id']).reset_index(drop=True))
display(DisplayImage(filename=str(histogram)))
print('Inventory checkpoint:', inventory_summary['collected_episodes'], 'collected episodes')
if inventory_summary['missing_episodes']:
    print('Missing from this input:', inventory_summary['missing_episodes'])
    print('Episode selection is restricted to collected episodes; no global-minimum claim')
environments = select_environments(qa_rows, config, inventory=inventory)
print('Selected episode:', environments[0], config['dataset']['environment_selection'])


## Sample frames
Keep one midpoint per disjoint sequence window, up to 200 frames. No capture timestamps are assumed.


In [ ]:
# Presentation: disjoint windows preserve coverage while capping the gallery at 200 frames.
rgb_root = acquire_rgb(config, WORK, archive_info, environments, source=source)
paths_by_env = discover_rgb(rgb_root, environments)
counted = next(row['frame_count'] for row in inventory if row['environment_id'] == environments[0])
if len(paths_by_env[environments[0]]) != counted:
    raise ValueError('Extracted RGB count differs from inventory; input changed or extraction is stale')
inspection = inspect_images(paths_by_env)
display(pd.DataFrame(inspection))
if any(row['invalid'] for row in inspection):
    raise ValueError('Invalid source images in the selected episode; inspect/repair data, no silent substitution')
frames, questions = prepare_gallery(paths_by_env, qa_rows, environments, config, BUNDLE)
atomic_json(BUNDLE/'provenance/acquisition.json', archive_info)
atomic_json(BUNDLE/'provenance/inspection.json', inspection)
display(pd.DataFrame([f.model_dump() for f in frames]).groupby(['environment_id']).size().to_frame('selected'))
display(pd.DataFrame([q.model_dump() for q in questions]).groupby(['category']).size().to_frame('questions'))
for env in environments:
    display(contact_sheet([f for f in frames if f.environment_id == env][:12], BUNDLE, title=env))
print('Gallery checkpoint:', len(frames), 'images; aspect-preserving JPEG preprocessing recorded')


## Review the video
Compare sampled and original views before enabling paid extraction. Playback FPS is only a review speed.


In [ ]:
sampled_video = make_preview_video([BUNDLE/f.image_relpath for f in frames], BUNDLE/'review/sampled_frames.mp4',
                                  sequence_indexes=[f.sequence_index for f in frames],
                                  fps=config['dataset']['preview_fps'], synthetic=SMOKE)
original_video = make_preview_video(paths_by_env[environments[0]], WORK/'original_episode_preview.mp4',
                                   fps=config['dataset']['preview_fps'], synthetic=SMOKE)
print('Sampled gallery:', len(frames), 'of', counted, 'frames')
print('All source frames retained' if len(frames) == counted else 'Review sampling loss against the original sequence below')
display(Video(str(sampled_video), embed=True, width=720))
display(Video(str(original_video), embed=True, width=720))
print('Sampled video exported in bundle/review; original video remains in WORK only')


## Inspect three extractions
Review the exact input images alongside captions, objects and relationships before proceeding.


In [ ]:
# Presentation: each image produces a caption and a frame-local scene graph.
from rvm.extraction import StructuredRunner, extract_frames, EXTRACTION_PROMPT, usage_summary
from rvm.io import file_hash
CACHE = WORK/'extraction-cache'
selection = read_json(BUNDLE/'selection.json')
preprocessing = selection.get('preprocessing', {'synthetic': True})
pilot_frames = [f for f in frames if f.frame_id in selection['pilot_frame_ids']]
if SMOKE:
    from rvm.fixture import complete_prepared_fixture
    complete_prepared_fixture(BUNDLE)
    pilot_records = [Record.model_validate(r) for r in read_json(BUNDLE/'records.json')
                     if r['frame_id'] in {f.frame_id for f in pilot_frames}]
else:
    if not SAMPLING_REVIEWED:
        raise RuntimeError('Review the sampled/original videos above, then set SAMPLING_REVIEWED=True before paid extraction')
    if not RUN_PILOT:
        raise RuntimeError('Preview videos are ready. Set RUN_PILOT=True to process the three-frame preview (all if fewer).')
    runner = StructuredRunner(CACHE, config['extraction'])
    pilot_records = extract_frames(pilot_frames, BUNDLE, runner, preprocessing=preprocessing, show_progress=True)
from IPython.display import HTML
from rvm.review import extraction_review_html
preview = extraction_review_html(pilot_frames, pilot_records, BUNDLE,
                                model=config['extraction']['model'], synthetic=SMOKE)
REVIEW = WORK/'pilot-review'
REVIEW.mkdir(parents=True, exist_ok=True)
(REVIEW/'extractions.html').write_text('<!doctype html><html><head><meta charset="utf-8">'
    '<meta name="viewport" content="width=device-width, initial-scale=1">'
    '<title>Pilot extraction review</title></head><body>' + preview + '</body></html>', encoding='utf-8')
atomic_json(REVIEW/'records.json', [r.model_dump(mode='json') for r in pilot_records])
display(HTML(preview))
print('Review report:', REVIEW/'extractions.html')
print('Review all three image/output pairs before setting PILOT_REVIEWED=True. Full extraction remains separately gated.')
print('Pilot checkpoint: review these records before enabling full extraction')


## Extract the gallery
Reuse cached pilot results and save completed frame records. API requests require the explicit flags above.


In [ ]:
if not SMOKE:
    if not (PILOT_REVIEWED and RUN_FULL_EXTRACTION):
        raise RuntimeError('Review the extraction pilot, then explicitly enable PILOT_REVIEWED and RUN_FULL_EXTRACTION')
    records = extract_frames(frames, BUNDLE, runner, preprocessing=preprocessing, show_progress=True)
    atomic_json(BUNDLE/'records.json', [r.model_dump(mode='json') for r in records])
    atomic_json(BUNDLE/'provenance/extraction-usage.json', usage_summary(CACHE))
    # All attempt records, including failed validation responses; never request headers/API keys.
    for attempt in tqdm(list((CACHE/'attempts').glob('*.json')), desc='Save request provenance', unit='attempt'):
        atomic_json(BUNDLE/'provenance/attempts'/attempt.name, read_json(attempt))
print('Extraction checkpoint:', len(read_json(BUNDLE/'records.json')), 'validated frame records')


## Export
Validate and package the graph bundle for local Neo4j import.


In [ ]:
from rvm.bundle import finalize_bundle
validated = finalize_bundle(BUNDLE, synthetic=SMOKE, show_progress=True)
ZIP = export_zip(BUNDLE, WORK/'robot-visual-memory-compact.zip', show_progress=True)
print('Export checkpoint:', ZIP, round(ZIP.stat().st_size/1024**2, 2), 'MiB')
print('ZIP was extracted and revalidated in a fresh temporary directory. Download only this compact ZIP.')
